# 作业一：心电数据分类（实验 notebook）
演示数据加载、补零处理对比（统一长度 vs 裁剪尾部 0）、特征提取与基线模型评估。

In [1]:
# 基本环境与模块路径设置
import sys
from pathlib import Path
root = Path('..').resolve()  # notebook 存在 notebooks/ 下，项目根为上级目录
sys.path.append(str(root))
print('root:', root)

root: /Data5/ddf/projects/2026sp_MLAI/hw_1


In [2]:
# 导入项目模块与常用包
import numpy as np
import pandas as pd
import importlib
import src.data.preprocessing as pp
importlib.reload(pp)
from src.data.preprocessing import load_csv, load_csv_robust, trim_trailing_zeros, pad_to_length, normalize_signals
from src.features.extractors import extract_features_list
from src.models.train import get_candidate_classifiers, evaluate_model
from src.evaluation.utils import plot_confusion, classification_metrics
print('imports ok')


imports ok


In [3]:
# 数据路径（如有必要请修改为你的路径）
data_path = str(root / 'data' / 'processed' / 'mitbih_train_downsampled_3000.csv')
print('data_path:', data_path)
X_raw, y = load_csv_robust(data_path)
print('raw X shape:', X_raw.shape)
unique, counts = np.unique(y, return_counts=True)
print('label distribution:', dict(zip(unique, counts)))


data_path: /Data5/ddf/projects/2026sp_MLAI/hw_1/data/processed/mitbih_train_downsampled_3000.csv
raw X shape: (2999, 318)
label distribution: {0: 2484, 1: 76, 2: 198, 3: 21, 4: 220}


## 策略 A：直接使用 CSV 内的统一长度（原始矩阵）作为输入并提取特征

In [4]:
# 直接使用原始固定长度信号（已被补零到统一长度）
X_fixed = X_raw.copy().astype(float)
X_fixed_norm = normalize_signals(X_fixed)
feats_fixed = extract_features_list([row for row in X_fixed_norm])
print('feats_fixed shape:', feats_fixed.shape)

feats_fixed shape: (2999, 15)


## 策略 B：裁剪尾部的 0（按每个样本的有效长度），再对裁剪后的信号 pad 到中位长度用于特征提取

In [5]:
trimmed_list, lengths = trim_trailing_zeros(X_raw)
print('example lengths (first 10):', lengths[:10])
# pad to median length for stable feature sizes
X_trim_padded = pad_to_length(trimmed_list, length=None)
X_trim_padded_norm = normalize_signals(X_trim_padded)
feats_trim = extract_features_list([row for row in X_trim_padded_norm])
print('feats_trim shape:', feats_trim.shape)

example lengths (first 10): [318 100 115 109 119  95 120 141 100 102]
feats_trim shape: (2999, 15)


## 基线模型评估示例（RandomForest），使用 stratified 5-fold CV 对比两种处理方式的特征表现

In [6]:
clfs = get_candidate_classifiers()
clf = clfs['rf']
res_fixed = evaluate_model(feats_fixed, y, clf, cv=5)
res_trim = evaluate_model(feats_trim, y, clf, cv=5)
print('fixed mean f1_macro:', res_fixed['test_f1_macro'].mean())
print('trimmed mean f1_macro:', res_trim['test_f1_macro'].mean())
print('fixed all scores keys:', list(res_fixed.keys()))

/home/ddf/miniconda3/envs/mlai/lib/python3.6/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ddf/miniconda3/envs/mlai/lib/python3.6/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ddf/miniconda3/envs/mlai/lib/python3.6/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ddf/miniconda3/envs/mlai/lib/python3.6/s

fixed mean f1_macro: 0.6619051273549748
trimmed mean f1_macro: 0.5644753581061224
fixed all scores keys: ['fit_time', 'score_time', 'test_accuracy', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro']


/home/ddf/miniconda3/envs/mlai/lib/python3.6/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## 下一步建议
- 在此基础上加入 SMOTE（`imblearn.over_sampling.SMOTE`）或直接用 `class_weight` 比较不平衡处理效果。
- 比较更多模型（`logreg`, `xgb`, `lgbm`），绘制混淆矩阵与 PR 曲线。
- 将关键函数保存在 `src/` 中以便复用，Notebook 保持可读的实验流程。